# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [34]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A line chart was selected because it effectively shows how population growth has changed over time across all six continents. Population growth is calculated relative to the 1970 baseline, allowing a fair comparison between continents with different initial population sizes. Africa is highlighted in blue while the other continents are shown in grey to make the main finding immediately visible.

In [35]:
continent_pop = (
    df_long.groupby(['Continent', 'Year'])['Population']
    .sum()
    .reset_index()
)

baseline = (
    continent_pop[continent_pop['Year'] == 1970]
    [['Continent', 'Population']]
    .rename(columns={'Population': 'Population_1970'})
)


continent_pop = continent_pop.merge(baseline, on='Continent')


continent_pop['Growth (%)'] = (
    (continent_pop['Population'] / continent_pop['Population_1970']) - 1
) * 100


colors = {
    'Africa': '#0072B2',
    'Asia': '#CCCCCC',
    'Europe': '#CCCCCC',
    'North America': '#CCCCCC',
    'South America': '#CCCCCC',
    'Oceania': '#CCCCCC'
}

fig = px.line(
    continent_pop,
    x='Year',
    y='Growth (%)',
    color='Continent',
    color_discrete_map=colors,
    template='plotly_white',
    title="Africa's Population Has Grown the Fastest Since 1970",
    labels={
        'Year': '',
        'Growth (%)': 'Population growth since 1970 (%)'
    }
)

fig.update_layout(
    showlegend=False,
    yaxis=dict(showgrid=True, gridcolor='#EEEEEE'),
    xaxis=dict(showgrid=False),
    title_font_size=20,
    margin=dict(r=100)
)

fig.update_traces(line=dict(width=1.5))

fig.update_traces(
    line=dict(width=3),
    selector=dict(name='Africa')
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a horizontal bar chart because it makes comparing population increases across 15 countries easy. The countries are sorted in descending order, allowing viewers to quickly identify those that added the most people between 1970 and 2022.

I used blue to highlight African countries and grey for the remaining countries to make the regional comparison clear. The chart shows that India and China experienced the largest population increases. Four of the top 15 countries are in Africa: Nigeria, Ethiopia, DR Congo and Egypt.

In [36]:
df['Population Added'] = (
    df['2022 Population'] - df['1970 Population']
)

top15 = df.nlargest(15, 'Population Added').copy()


africa_count = (top15['Continent'] == 'Africa').sum()


top15 = top15.sort_values('Population Added')


bar_colors = [
    '#0072B2' if continent == 'Africa' else '#CCCCCC'
    for continent in top15['Continent']
]


fig = px.bar(
    top15,
    x='Population Added',
    y='Country',
    orientation='h',
    text='Population Added',
    template='plotly_white',
    title=f'Top 15 Countries Added the Most People: {africa_count} Are in Africa',
    labels={
        'Population Added': 'Additional population (people)',
        'Country': ''
    }
)


fig.update_traces(
    marker_color=bar_colors,
    texttemplate='%{text:,.0f}',
    textposition='outside',
    cliponaxis=False
)

fig.update_layout(
    showlegend=False,
    xaxis=dict(
        showgrid=True,
        gridcolor='#EEEEEE',
        tickformat=',.0f'
    ),
    yaxis=dict(
        showgrid=False,
        categoryorder='array',
        categoryarray=top15['Country'].tolist()
    ),
    title_font_size=20,
    margin=dict(r=130)
)

fig.show()

print("African countries among the top 15:", africa_count)

print(top15[['Country', 'Continent', 'Population Added']].sort_values(
    'Population Added', ascending=False
).to_string(index=False))

African countries among the top 15: 4
      Country     Continent  Population Added
        India          Asia         859671872
        China          Asia         603352887
     Pakistan          Asia         176533990
      Nigeria        Africa         162971948
    Indonesia          Asia         160272945
United States North America         137961517
       Brazil South America         118943623
   Bangladesh          Asia         103644512
     Ethiopia        Africa          95071678
     DR Congo        Africa          78858479
  Philippines          Asia          78123423
       Mexico North America          77214819
        Egypt        Africa          76208117
         Iran          Asia          60100865
      Vietnam          Asia          56258007


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a scatter plot because it helps visualise the relationship between a country's land area and its population in 2022. Each dot represents a country, and different colours represent different continents. I used logarithmic scales on both axes because the values vary significantly between countries.

The chart shows a general positive relationship between land area and population, but it is not consistent across all countries. India and Bangladesh have relatively high populations compared to their land areas, whereas Canada, Australia and Russia have large land areas but comparatively smaller populations.

This suggests that land area alone does not determine population size. Population distribution also varies considerably between continents.

In [37]:
import plotly.express as px
import pandas as pd

q3_df = df[['Country', 'Continent', 'Area', '2022 Population']].copy()

q3_df = q3_df.dropna()

q3_df = q3_df[
    (q3_df['Area'] > 0) &
    (q3_df['2022 Population'] > 0)
]

fig = px.scatter(
    q3_df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    hover_data={
        'Area': ':,',
        '2022 Population': ':,'
    },
    log_x=True,
    log_y=True,
    title='Does a Bigger Country Have More People?',
    labels={
        'Area': 'Land Area (km²)',
        '2022 Population': 'Population (2022)',
        'Continent': 'Continent'
    },
    template='plotly_white',
    height=600
)

fig.update_traces(
    marker=dict(size=10, opacity=0.75)
)

fig.update_layout(
    title_font_size=20,
    xaxis=dict(showgrid=True, gridcolor='#EEEEEE'),
    yaxis=dict(showgrid=True, gridcolor='#EEEEEE'),
    legend_title_text='Continent'
)

fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a box plot combined with individual data points to compare population density across continents.

The box plots display the median, interquartile range and distribution of population density, while individual points help identify outliers and differences between countries.

I used a logarithmic Y-axis because population density varies significantly across countries. This makes it easier to compare countries with both high and low population densities.

The chart indicates that North America has the highest median population density, while South America has the lowest. Asia and Europe also show relatively high population density, whereas Africa displays considerable variation among its countries.

In [38]:
import plotly.express as px
import pandas as pd

q4_df = df[['Country', 'Continent', 'Density']].copy()

q4_df = q4_df.dropna()

q4_df = q4_df[q4_df['Density'] > 0]

fig = px.box(
    q4_df,
    x='Continent',
    y='Density',
    color='Continent',
    points='all',
    hover_name='Country',
    log_y=True,
    title='Population Density Varies Across Continents',
    labels={
        'Continent': 'Continent',
        'Density': 'Population Density (people per km²)'
    },
    template='plotly_white',
    height=600
)

fig.update_traces(
    marker=dict(size=5, opacity=0.65),
    boxmean=True
)

fig.update_layout(
    title_font_size=20,
    showlegend=False,
    xaxis=dict(showgrid=False),
 yaxis=dict(
    type='log',
    title='Population Density (people per km²)',
    tickmode='array',
    tickvals=[
        0.1, 0.2, 0.5,
        1, 2, 5,
        10, 20, 50,
        100, 200, 500,
        1000, 2000, 5000,
        10000, 20000
    ],
    ticktext=[
        '0.1', '0.2', '0.5',
        '1', '2', '5',
        '10', '20', '50',
        '100', '200', '500',
        '1,000', '2,000', '5,000',
        '10,000', '20,000'
    ],
    showgrid=True,
    gridcolor='#EEEEEE'
)
)

fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose an interactive choropleth map to visualise population percentage changes across countries between 2000 and 2022.

The map uses a diverging colour scale to distinguish countries experiencing population growth from those experiencing population decline. Green represents positive growth, while red represents negative growth.

I calculated percentage change rather than absolute population change to make comparisons between countries of different population sizes more meaningful.

The interactive map also allows users to hover over individual countries to examine their population figures and percentage changes.

In [39]:
import pandas as pd
import plotly.express as px

population_years = df_long[df_long['Year'].isin([2000, 2022])]

population_change = population_years.pivot_table(
    index=['CCA3', 'Country', 'Continent'],
    columns='Year',
    values='Population'
).reset_index()

population_change = population_change.rename(columns={
    2000: 'Population_2000',
    2022: 'Population_2022'
})

population_change['Percentage_Change'] = (
    (population_change['Population_2022'] -
     population_change['Population_2000'])
    / population_change['Population_2000']
) * 100

population_change = population_change.dropna(
    subset=['Percentage_Change']
)

print("Top 10 countries with population growth:")
print(
    population_change.nlargest(10, 'Percentage_Change')[
        ['Country', 'Continent', 'Percentage_Change']
    ].to_string(index=False)
)

print("\nTop 10 countries with population decline:")
print(
    population_change.nsmallest(10, 'Percentage_Change')[
        ['Country', 'Continent', 'Percentage_Change']
    ].to_string(index=False)
)

Top 10 countries with population growth:
                 Country     Continent  Percentage_Change
                   Qatar          Asia         317.242239
    United Arab Emirates          Asia         188.249439
       Equatorial Guinea        Africa         144.520327
Turks and Caicos Islands North America         143.827358
                   Niger        Africa         125.490255
                  Jordan          Asia         123.209664
                  Kuwait          Asia         120.624880
                  Angola        Africa         117.084619
                    Chad        Africa         114.590398
          Western Sahara        Africa         113.032270

Top 10 countries with population decline:
                 Country Continent  Percentage_Change
Northern Mariana Islands   Oceania         -38.321840
          American Samoa   Oceania         -23.968745
               Lithuania    Europe         -23.601880
        Marshall Islands   Oceania         -23.338374
        

In [40]:
fig = px.choropleth(
    population_change,
    locations='Country',
    locationmode='country names',
    color='Percentage_Change',
    hover_name='Country',
    hover_data={
        'Continent': True,
        'Population_2000': ':,',
        'Population_2022': ':,',
        'Percentage_Change': ':.2f'
    },
    color_continuous_scale='RdYlGn',
    range_color=(-40, 320),
    title='Population Growth and Decline Across the World (2000–2022)',
    labels={
        'Percentage_Change': 'Population Change (%)'
    },
    template='plotly_white'
)

fig.update_layout(
    height=600,
    title_x=0.05,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    )
)

fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a treemap because it effectively displays hierarchical population data at two levels: continents and individual countries. Each continent is represented as a larger section, with individual countries nested inside it. The size of each rectangle represents its share of the 2022 world population. This makes it easy to compare population distribution across continents and identify the countries with the largest populations. The interactive features also allow users to explore individual countries and view their population figures.

In [41]:
import plotly.express as px

df_2022 = df_long[df_long['Year'] == 2022].copy()

fig = px.treemap(
    df_2022,
    path=['Continent', 'Country'],
    values='Population',
    color='Population',
    color_continuous_scale='Blues',
    hover_data={
        'Population': ':,'
    },
    title="Where Do the World's 8 Billion People Live? (2022)"
)

fig.update_layout(
    height=750,
    title_x=0.05,
    margin=dict(t=80, l=20, r=20, b=20)
)

fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

I chose a 100% stacked area chart because it clearly illustrates how the share of world population has changed across continents between 1970 and 2022. It allows us to compare the relative contribution of each group over time.

Africa experienced the largest increase, rising from 9.89% in 1970 to 17.89% in 2022 (+8.00 percentage points). Asia remained the largest contributor, increasing slightly from 58.06% to 59.21%. In contrast, Europe's share declined significantly from 17.76% to 9.32% (-8.44 percentage points). The rest of the world also experienced a small decline from 14.29% to 13.57%.

Overall, the chart shows that Africa's share of the global population is growing rapidly, while Europe's relative share is decreasing.

In [42]:
import pandas as pd
import plotly.express as px

share_df = df_long.copy()

share_df['Group'] = share_df['Continent'].replace({
    'North America': 'Rest of the world',
    'South America': 'Rest of the world',
    'Oceania': 'Rest of the world'
})

grouped = share_df.groupby(
    ['Year', 'Group'], as_index=False
)['Population'].sum()

grouped['World Share (%)'] = (
    grouped['Population'] /
    grouped.groupby('Year')['Population'].transform('sum')
) * 100

fig = px.area(
    grouped,
    x='Year',
    y='World Share (%)',
    color='Group',
    category_orders={
        'Group': [
            'Africa',
            'Asia',
            'Europe',
            'Rest of the world'
        ]
    },
    title='Changing Share of World Population (1970–2022)',
    labels={
        'Year': 'Year',
        'World Share (%)': 'Share of World Population (%)',
        'Group': 'Continent'
    }
)

fig.update_layout(
    height=650,
    yaxis=dict(range=[0, 100]),
    hovermode='x unified',
    title_x=0.05
)

fig.show()

comparison = grouped[
    grouped['Year'].isin([1970, 2022])
].pivot(
    index='Group',
    columns='Year',
    values='World Share (%)'
)

comparison['Change (percentage points)'] = (
    comparison[2022] - comparison[1970]
)

print("Population share comparison (1970 vs 2022):")
print(comparison.round(2))

Population share comparison (1970 vs 2022):
Year                1970   2022  Change (percentage points)
Group                                                      
Africa              9.89  17.89                        8.00
Asia               58.06  59.21                        1.15
Europe             17.76   9.32                       -8.44
Rest of the world  14.29  13.57                       -0.72
